Here’s a basic sanity-check script — the idea is to glitch across a range of width/offset values during reset release and watch for a change in the SAM4S’s boot behavior (crash, no response, or corrupted output vs. a clean boot), which tells you the crowbar is physically having an effect on the chip.

In [6]:
import chipwhisperer as cw
import time

try:
    scope.dis()
except:
    pass

scope = cw.scope()
target = cw.target(scope, cw.targets.SimpleSerial)  # adjust if your SAM4S target uses a different comms class

# --- Clock setup ---
scope.clock.clkgen_freq = 7.37e6
scope.clock.adc_mul = 1
time.sleep(0.5)
assert scope.clock.clkgen_locked, "clkgen did not lock"

# --- Glitch module setup ---
scope.glitch.enabled = False
time.sleep(0.2)
scope.glitch.enabled = True
time.sleep(0.5)
scope.glitch.clk_src = 'pll'
scope.clock.fpga_vco_freq = 600e6
time.sleep(0.5)
assert scope.glitch.mmcm_locked, "glitch MMCM did not lock"

scope.glitch.output = "glitch_only"
scope.glitch.repeat = 1

# --- Trigger setup: nRSTOUT via CW313 JP2 -> GPIO4/tio4 ---
scope.io.tio3 = "high_z"
scope.trigger.triggers = "tio3"
scope.adc.basic_mode = "rising_edge"
scope.glitch.trigger_src = "ext_single"

# --- Crowbar MOSFET paths ---
scope.io.glitch_hp = True
scope.io.glitch_lp = True


def reset_target():
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high_z'
    time.sleep(0.05)


def try_glitch(width, offset):
    scope.glitch.width = width
    scope.glitch.offset = offset

    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.arm()
    scope.io.nrst = 'high'

    ret = scope.capture()
    timeout = ret  # True = ADC capture timed out waiting for trigger

    # Give the target a moment to respond post-reset
    time.sleep(0.05)
    resp = target.read() if hasattr(target, "read") else None

    return timeout, resp


# --- Baseline: no glitch, confirm normal boot response ---
print("=== Baseline (glitch disabled) ===")
scope.io.glitch_hp = False
scope.io.glitch_lp = False
reset_target()
time.sleep(0.1)
baseline_resp = target.read() if hasattr(target, "read") else None
print("Baseline response:", repr(baseline_resp))

scope.io.glitch_hp = True
scope.io.glitch_lp = True

# --- Sweep a simple width/offset range ---
print("\n=== Glitch sweep ===")
results = []
for width in range(5, 50, 5):
    for offset in range(-20, 20, 5):
        timeout, resp = try_glitch(width, offset)
        changed = (resp != baseline_resp)
        results.append((width, offset, timeout, resp, changed))
        flag = "CHANGED" if changed else ("TIMEOUT" if timeout else "normal")
        print(f"width={width:3d} offset={offset:4d} -> {flag} resp={resp!r}")

print("\n=== Summary ===")
hits = [r for r in results if r[4] or r[2]]
print(f"{len(hits)} / {len(results)} parameter combos showed a deviation from baseline")

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


=== Baseline (glitch disabled) ===
Baseline response: ''

=== Glitch sweep ===
width=  5 offset= -20 -> normal resp=''
width=  5 offset= -15 -> normal resp=''
width=  5 offset= -10 -> normal resp=''
width=  5 offset=  -5 -> normal resp=''
width=  5 offset=   0 -> normal resp=''
width=  5 offset=   5 -> normal resp=''
width=  5 offset=  10 -> normal resp=''
width=  5 offset=  15 -> normal resp=''
width= 10 offset= -20 -> normal resp=''
width= 10 offset= -15 -> normal resp=''
width= 10 offset= -10 -> normal resp=''
width= 10 offset=  -5 -> normal resp=''
width= 10 offset=   0 -> normal resp=''
width= 10 offset=   5 -> normal resp=''
width= 10 offset=  10 -> normal resp=''
width= 10 offset=  15 -> normal resp=''
width= 15 offset= -20 -> normal resp=''
width= 15 offset= -15 -> normal resp=''
width= 15 offset= -10 -> normal resp=''
width= 15 offset=  -5 -> normal resp=''
width= 15 offset=   0 -> normal resp=''
width= 15 offset=   5 -> normal resp=''
width= 15 offset=  10 -> normal resp=''
w

KeyboardInterrupt: 

Notes:

- target.read() / cw.targets.SimpleSerial assumes your SAM4S firmware talks SimpleSerial over UART — swap for whatever comms your actual target firmware uses (plain UART read, target.simpleserial_read, etc.). If you’re just checking “did it crash/reset weirdly,” even reading raw UART bytes with target.read() or a basic pyserial read is enough for a first pass.
- A “TIMEOUT” or garbled/empty response where baseline gave a clean one is good evidence the crowbar pulse disrupted execution.
- A crash with no response at all across a block of width/offset values (rather than scattered single hits) is a strong signal the glitch is physically landing — true single-cycle flips are usually sparser and more “needle in haystack.”
- Keep the sweep narrow at first (as above) just to prove the mechanism works, before doing a fine-grained scan for an actual effect you’re trying to induce (e.g., skipping a specific check).

What’s your target firmware’s comms setup — plain UART echo, SimpleSerial protocol, or something custom? I can tighten the read/response check to match.

Much simpler then — you just need the glitch to fire repeatedly on a predictable trigger so you can park your scope probe on the crowbar output and look for the voltage dip.

In [9]:
# script to check if nRST is connected with GPIO3
scope.io.tio3 = "high_z"
scope.io.nrst = 'low'
time.sleep(0.05)
print("low:", scope.io.tio_states)
scope.io.nrst = 'high'
time.sleep(0.05)
print("high:", scope.io.tio_states)

low: (1, 1, 0, 0)
high: (1, 1, 1, 0)


In [13]:
import chipwhisperer as cw
import time

try:
    scope.dis()
except:
    pass

scope = cw.scope()

# --- Clock setup ---
scope.clock.clkgen_freq = 7.37e6
scope.clock.adc_mul = 1
time.sleep(0.5)
assert scope.clock.clkgen_locked, "clkgen did not lock"

# --- Glitch module setup ---
scope.glitch.enabled = False
time.sleep(0.2)
scope.glitch.enabled = True
time.sleep(0.5)
scope.glitch.clk_src = 'pll'
scope.clock.fpga_vco_freq = 600e6
time.sleep(0.5)
assert scope.glitch.mmcm_locked, "glitch MMCM did not lock"

scope.glitch.output = "glitch_only"
scope.glitch.width = 10
scope.glitch.offset = 0
scope.glitch.repeat = 1

# --- Trigger: nRSTOUT via CW313 JP2 -> GPIO4/tio3 ---
scope.io.tio3 = "high_z"
scope.trigger.triggers = "tio3"
scope.adc.basic_mode = "rising_edge"
scope.glitch.trigger_src = "ext_single"

# --- Crowbar MOSFET paths ---
#scope.io.glitch_hp = True
#scope.io.glitch_lp = True
scope.io.glitch_hp = False
scope.io.glitch_lp = False

print(scope.glitch)

# --- Fire repeatedly so you can see it live on the scope ---
print("Probe the crowbar/SMA output now. Firing glitches every 1s, Ctrl+C to stop.")
try:
    while True:
        scope.io.nrst = 'low'
        time.sleep(0.05)
        scope.arm()
        scope.io.nrst = 'high'
        ret = scope.capture()
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopped.")

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:704) Target clock may drop; you may need to reset your target.


enabled           = True
num_glitches      = 1
clk_src           = pll
mmcm_locked       = True
width             = 10
offset            = 0
trigger_src       = ext_single
arm_timing        = after_scope
ext_offset        = 0
repeat            = 1
output            = glitch_only
phase_shift_steps = 4592

Probe the crowbar/SMA output now. Firing glitches every 1s, Ctrl+C to stop.
Stopped.


Probe setup:

- Put your scope probe on the target’s VCC rail right at the SAM4S (or on the CW313’s SMA glitch connector if you’re feeding the crowbar output there) — ground clip close by to avoid ringing.
- Set your bench scope to trigger on the falling edge of the same signal, or just free-run at a few hundred ns/div to 1 µs/div timebase since width=10 cycles at 7.37 MHz is roughly 1.3 µs — you want to see a sharp dip/short to ground during that window.
- If you see nothing, try increasing scope.glitch.width to something larger (e.g. 100) temporarily just to make the pulse easier to spot on screen, then dial back down once confirmed.

This loop just reset-triggers and fires once per second indefinitely until you Ctrl+C it, so you have time to adjust your scope settings live.